In [ ]:
# GOES soft X-ray + GOES integral proton flux, with flare peak times marked
# Based on supervisor's AR 10930 script; SEPEM data replaced by GOES integral proton channels.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import sunpy.timeseries
from sunpy.net import Fido
from sunpy.net import attrs as a
from sunpy.time import parse_time

In [ ]:
# ============================================================
# Settings
# ============================================================
START = pd.Timestamp("2021-12-04 00:00")
END   = pd.Timestamp("2021-12-06 00:00")     # exclusive end
RES   = "5min"                              # proton averaging resolution: 5min keeps everything in file, no smoothing

FLARE_THRESHOLD = "M1.0"   # supervisor used >X1.0; lower it (e.g. "M1.0", "C1.0") if the window has no X-flares

DIR_GOES  = Path(r"C:\Users\pauls\SEPs\coseecat_dr1\data\05122021\GOES")
GOES_FILE = sorted(DIR_GOES.glob("GOES_all_channels_*.csv"))[-1]
OUT_DIR = Path(r"C:\Users\pauls\OneDrive\Documents\Naomi\Lancs\coseecat_dr1_figs\04_05122021")
OUT_DIR.mkdir(parents=True, exist_ok=True)   # creates the folder if needed

goes_channels = {            # column name : legend label
    "P10": "GOES P10 (>10 MeV protons)",
    "P30": "GOES P30 (>30 MeV protons)",
    "P50": "GOES P50 (>50 MeV protons)",
}
PROTON_UNITS = r"cm$^{-2}$ s$^{-1}$ sr$^{-1}$"        


In [ ]:
# ============================================================
# GOES soft X-ray data (downloaded with Fido)
# ============================================================
tr = a.Time(START.to_pydatetime(), END.to_pydatetime())

goes_search = Fido.search(tr, a.Instrument.xrs)
goes_files = Fido.fetch(goes_search)
combined_goes_ts = sunpy.timeseries.TimeSeries(goes_files, source="XRS", concatenate=True)

In [ ]:
# ============================================================
# Flare list from HEK
# ============================================================
flare_results = Fido.search(
    tr,
    a.hek.EventType("FL"),
    a.hek.FL.GOESCls > FLARE_THRESHOLD,
    a.hek.OBS.Observatory == "GOES",
)

hek = flare_results["hek"] if "hek" in flare_results.keys() else None

if hek is not None and len(hek) > 0:
    print(hek.colnames)

    def find_col(keyword):
        return next((c for c in hek.colnames if keyword in c.lower()), None)

    # choose the peak *time* column explicitly, not just anything containing "peak"
    time_cols = [c for c in hek.colnames if "peak" in c.lower() and "time" in c.lower()]
    peak_col = "event_peaktime" if "event_peaktime" in hek.colnames else (time_cols[0] if time_cols else None)

    class_col = find_col("goescls")
    ar_col    = find_col("noaanum")

    if peak_col is None:
        raise KeyError(f"No peak-time column found. Columns are: {hek.colnames}")

    show_cols = [c for c in (peak_col, class_col, ar_col) if c is not None]
    print("FLARES:")
    print(hek[show_cols])
    print("n_flares =", len(hek))

    print("peak_col =", peak_col)        # temporary debugging
    print(hek[peak_col])                 # temporary debugging

    # skip any masked (missing) entries instead of crashing on them
    flare_peaks = [parse_time(str(t)).datetime
                   for t in hek[peak_col] if not np.ma.is_masked(t)]
    print(f"{len(flare_peaks)} of {len(hek)} flares have a peak time")
else:
    print(f"No flares above {FLARE_THRESHOLD} found in this window.")
    flare_peaks = []


In [ ]:
# ============================================================
# GOES integral proton data (from your CSV)
# ============================================================
goes = pd.read_csv(GOES_FILE)
missing = [c for c in goes_channels if c not in goes.columns]
if missing:
    raise KeyError(f"Not found: {missing}. Columns are: {list(goes.columns)}")

# Parse as UTC, then drop the tz so it matches the naive START/END
goes["Time"] = pd.to_datetime(goes["Time"], utc=True).dt.tz_localize(None)
goes = goes.set_index("Time").sort_index()[list(goes_channels)]
goes = goes.where(goes > 0)                  # zeros/negatives -> NaN for the log axis

goes10 = goes.resample(RES, closed="left", label="left").mean()
goes10 = goes10[(goes10.index >= START) & (goes10.index < END)]   # true exclusive end
print(goes10.notna().sum())

In [ ]:
# ============================================================
# GOES integral proton data (from your CSV)
# ============================================================
goes = pd.read_csv(GOES_FILE)
missing = [c for c in goes_channels if c not in goes.columns]
if missing:
    raise KeyError(f"Not found: {missing}. Columns are: {list(goes.columns)}")

# Parse as UTC, then drop the tz so it matches the naive START/END
goes["Time"] = pd.to_datetime(goes["Time"], utc=True).dt.tz_localize(None)
goes = goes.set_index("Time").sort_index()[list(goes_channels)]
goes = goes.where(goes > 0)                  # zeros/negatives -> NaN for the log axis

goes10 = goes.resample(RES, closed="left", label="left").mean()
goes10 = goes10[(goes10.index >= START) & (goes10.index < END)]   # true exclusive end
print(goes10.notna().sum())

In [ ]:
# ============================================================
# Plot
# ============================================================
fig, (ax0, ax1) = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(14, 8))
fig.subplots_adjust(hspace=0.08, right=0.78)

# Top panel: SXR 1-8 A channel
combined_goes_ts.plot(axes=ax0, columns=["xrsb"])
ax0.set_ylabel(r"SXR Flux [W m$^{-2}$]")
ax0.set_ylim(1e-8, 1e-2)
ax0.set_xlim(START, END)

# Bottom panel: GOES integral proton flux
for col, label in goes_channels.items():
    ax1.plot(goes10.index, goes10[col], label=label)

# Flare peak-time lines on both panels (label only the first so the legend has one entry)
flare_label = f"Flare peak (> {FLARE_THRESHOLD})"
for i, t in enumerate(flare_peaks):
    lab = flare_label if i == 0 else None
    ax0.axvline(t, color="grey", alpha=0.7, label=lab)
    ax1.axvline(t, color="grey", alpha=0.7, label=lab)

ax0.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
ax1.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0)

ax1.set_yscale("log")

# year(s) covered by the plot, e.g. "2021" or "2021-2022" if it crosses New Year
last = END - pd.Timedelta(seconds=1)
years = sorted({START.year, last.year})
year_str = "-".join(str(y) for y in years)

ax1.set_xlabel(f"Time (UTC), {year_str}")

ax1.xaxis.get_offset_text().set_visible(False)
ax1.set_ylabel(rf"Proton flux [{PROTON_UNITS}]")

# Flare peak-time lines on both panels
for t in flare_peaks:
    ax0.axvline(t, color="grey", alpha=0.7)
    ax1.axvline(t, color="grey", alpha=0.7)

plt.savefig(OUT_DIR / "goes_flare_proton_05122021_v2.jpg", dpi=400, bbox_inches="tight")
plt.show()